In [ ]:
from pathlib import Path
Path('../outputs').mkdir(parents=True, exist_ok=True)


# 라이브러리 불러오기

In [ ]:
import pandas as pd
import numpy as np
import seaborn as sns
import matplotlib.pyplot as plt
import scipy.stats as stats
import warnings
warnings.filterwarnings('ignore')

# 데이터 불러오기

In [ ]:
train = pd.read_csv('../data/raw/train.csv')
test = pd.read_csv('../data/raw/test.csv')

In [ ]:
train.head()

In [ ]:
test.head()

# 결측치 확인

- train 데이터에는 결측치 없음

In [ ]:
print('train 결측치 개수:', train.isnull().sum().sum())
train.info()

- test 데이터에 결측치 하나 있음

In [ ]:
print('test 결측치 개수:', test.isnull().sum().sum())
test.info()

# 중복 확인 및 제거

In [ ]:
# ID 컬럼을 제외한 데이터에서 중복 행 개수 확인
train.iloc[:, 1:].duplicated().sum()

In [ ]:
# 중복인 행 직접 확인
train.iloc[:, 1:][train.iloc[:, 1:].duplicated(keep=False)].sort_values(by=['Age', 'Gender', 'Employment_Status'])

In [ ]:
# 중복인 행 제거
train = train.drop_duplicates(train.columns[1:], keep='first')
train

In [ ]:
# 중복 제거 잘 됐는지 확인
train.iloc[:, 1:].duplicated().sum()

# 변수 확인

## Income

- 종속변수
- 1시간당 근로소득

In [ ]:
train['Income'].value_counts()

### Income 값이 0인 행 탐색

- Income 값들 중 0이 많다 -> 한번 집중해보자!

In [ ]:
# income이 0인 행과 아닌 행을 분리
income_zero = train[train['Income'] == 0]
income_nonzero = train[train['Income'] != 0]

In [ ]:
# income이 0인 데이터만 가지는 class 확인 
for col in income_nonzero.columns[1:]:
    if income_nonzero[col].dtype == 'object':

        if set(income_nonzero[col].unique()) == set(income_zero[col].unique()):
            print(f'{col}의 income_nonzero, income_zero 범주가 같음')
        else:
            print(f'{col}의 income_nonzero, income_zero 범주가 다름')
            print(f'income_nonzero 범주 : {income_nonzero[col].nunique()}개, income_zero 범주 : {income_zero[col].nunique()}개')
            # 없는 범주 출력
            print(f'income_nonzero에만 있는 범주 : {set(income_nonzero[col].unique()) - set(income_zero[col].unique())}')
            print(f'income_zero에만 있는 범주: {set(income_zero[col].unique()) - set(income_nonzero[col].unique())}')
        print('-----------------')

In [ ]:
# Education_Status가 Children이면 Income이 0 -> 미리 처리
train[train['Education_Status'] == 'Children']['Income'].value_counts()

In [ ]:
# Employment_Status가 Not working이면 Income이 0 -> 미리 처리
train[train['Employment_Status'] == 'Not Working']['Income'].value_counts()

In [ ]:
# Industry_Status가 Not in universe or children이면 Income이 0 -> 미리 처리
train[train['Industry_Status'] == 'Not in universe or children']['Income'].value_counts()

In [ ]:
# Industry_Status가 Armed Forces이면 Income이 0 이지만 표본이 너무 적음 -> 보류
train[train['Industry_Status'] == 'Armed Forces']['Income'].value_counts()

In [ ]:
# Occupation_Status가 Armed Forces이면 Income이 0 이지만 표본이 너무 적음 -> 보류
# Industry_Status가 Armed Forces인 경우와 같은 행
train[train['Occupation_Status'] == 'Armed Forces']['Income'].value_counts()

In [ ]:
# Occupation_Status가 Unknown이면 Income이 0 -> 미리 처리
train[train['Occupation_Status'] == 'Unknown']['Income'].value_counts()

In [ ]:
# Birth_Country가 Panama 인경우 Income이 0 이지만 표본이 너무 적음 -> 보류
train[train['Birth_Country'] == 'Panama']['Income'].value_counts()

In [ ]:
# 마찬가지로 Birth_Country(Father)가 Panama 인경우 Income이 0 이지만 표본이 너무 적음 -> 보류
train[train['Birth_Country (Father)'] == 'Panama']['Income'].value_counts()

In [ ]:
# 14세 이하이면 Income이 0 -> 미리 처리
train[train['Age'] <= 14]['Income'].value_counts()

### Income 값이 확실히 0인것 제거

- 확실히 Income 값이 0인 경우
    - Education_Status 컬럼이 Children이다.
    - Employment_Status 컬럼이 Not working이다.
    - Industry_Status 컬럼이 Not in universe or children이다.
    - Occupation_Status 컬럼이 Unknown이다.
    - Age 컬럼이 14 이하이다.

- 확실히 income이 0인 행은 학습과 예측이 필요없으니 제거
- test 데이터에도 마찬가지로 적용

In [ ]:
# train 데이터에 적용
train = train[train['Education_Status'] != 'Children']
train = train[train['Employment_Status'] != 'Not Working']
train = train[train['Industry_Status'] != 'Not in universe or children']
train = train[train['Occupation_Status'] != 'Unknown']
train = train[train['Age'] > 14]
train

In [ ]:
# test 데이터에 적용
test = test[test['Education_Status'] != 'Children']
test = test[test['Employment_Status'] != 'Not Working']
test = test[test['Industry_Status'] != 'Not in universe or children']
test = test[test['Occupation_Status'] != 'Unknown']
test = test[test['Age'] > 14]
test

- 확실히 income이 0인 행을 분리하는 과정에서 test 데이터 속 결측치 행이 처리됨 -> 추가적인 결측치 처리 필요 없음

In [ ]:
print('test 결측치 개수:', test.isnull().sum().sum())
test.info()

## ID

- 유의미한 정보 아님 -> 학습 시 drop
- 나중에 test 데이터 제출 시 활용 정도만

In [ ]:
train['ID'].value_counts()

## Age

- 나이

In [ ]:
train['Age'].value_counts()

- 14세 이하는 없음 -->  확실히 0인 경우를 제거한 영향

In [ ]:
# age histogram
sns.histplot(train['Age'])

In [ ]:
# age boxplot
sns.boxplot(train['Age'])

In [ ]:
# age와 income 상관계수 
train['Age'].corr(train['Income'])

In [ ]:
# age와 income의 scatterplot
sns.scatterplot(x='Age', y='Income', data=train)

In [ ]:
# income이 0이 아닌 행에 대한 age와 income 상관계수 
train[train['Income'] > 0]['Age'].corr(train[train['Income'] > 0]['Income'])

In [ ]:
# income이 0이 아닌 행에 대한 age와 income의 scatterplot
sns.scatterplot(x='Age', y='Income', data=train[train['Income'] > 0])

## Gender

- 성별

In [ ]:
train['Gender'].value_counts()

In [ ]:
# gender와 income의 boxplot
sns.boxplot(x='Gender', y='Income', data=train, hue='Gender')

- 독립표본 t test 시행 시도
- 정규성 검정, 등분산 검정 시도 -> 기각 -> 비모수 방법인 mann whitney u 사용
- 귀무가설 기각(p value < 0.05) -> 두 집단은 Income에 대해 유의미한 차이가 있다.

In [ ]:
# 남성과 여성의 소득 데이터 추출
income_m = train[train['Gender'] == 'M']['Income']
income_f = train[train['Gender'] == 'F']['Income']

# 정규성 검정 -> p < 0.05 이므로 정규성 만족 X
print('M 집단 정규성 검정 결과 :', stats.shapiro(income_m))
print('F 집단 정규성 검정 결과 :', stats.shapiro(income_f))
print()

# 등분산 검정 -> p < 0.05 이므로 등분산 만족 X
print('M, F 집단 간 등분산성 검정 결과 :',stats.levene(income_m, income_f))
print()

# 비모수 검정 -> p < 0.05 이므로 두 집단의 소득 차이가 유의미함
print('mann whitney u 검정 결과 :',stats.mannwhitneyu(income_m, income_f))

## Education_Status

- 최종학력을 의미

- 'Children': '어린이'
- 'Kindergarten': '유치원'
- 'Elementary (1-4)': '초등학교 (1-4학년)'
- 'Elementary (5-6)': '초등학교 (5-6학년)'
- 'Middle (7-8)': '중학교 (7-8학년)'
- 'High Freshman': '고등학교 1학년'
- 'High Sophomore': '고등학교 2학년'
- 'High Junior': '고등학교 3학년'
- 'High Senior': '고등학교 4학년'
- 'High graduate': '고등학교 졸업'
- 'College': '대학'
- 'Associates degree (Vocational)': '전문학사 학위 (직업)'
- 'Associates degree (Academic)': '전문학사 학위 (학문)'
- 'Bachelors degree': '학사 학위'
- 'Professional degree': '전문 학위'
- 'Masters degree': '석사 학위'
- 'Doctorate degree': '박사 학위'

In [ ]:
train['Education_Status'].value_counts()

In [ ]:
# Education_Status와 Income의 boxplot
sns.boxplot(y='Education_Status', x='Income', data=train, hue='Education_Status')

### 범주들 그룹화

- 유사한 분포를 보이는 범주끼리 그룹화 해보자!
- 그 그룹화가 잘 되었는지 확인하기 위해서 통계적 검정 시행

    -> 그룹화 된 범주들 간의 Income 분포 차이가 있는지 확인(정규성을 만족 못한다고 판단하고 비모수 검정으로 진행)
    - 비교하는 범주가 2개면 mann whitney u 검정 시행
    - 비교하는 범주가 3개 이상이면 Kruskal-Wallis H 검정 시행    

- 모든 범주가 그룹화되는건 아님 -> 유사한 범주들만 그룹화 진행
- 16개 범주에서 9개 범주로 축소

In [ ]:
# 최종 결정된 그룹들
cluster_1 = ['Kindergarten', 'Elementary (1-4)', 'Elementary (5-6)']
cluster_2 = ['Middle (7-8)','High Freshman', 'High Sophomore','High Junior', 'High Senior']
cluster_3 = ['Professional degree', 'Masters degree']

In [ ]:
# 그룹 속 범주 간의 Income 차이 검정 -> 그룹 속 범주 간 Income 차이가 없음 
for cluster in [cluster_1, cluster_2, cluster_3]:
    print(f"Education_Status: {cluster}")
    if len(cluster) == 1:
        print('범주 1개라 시행안함')
        continue
    elif len(cluster) == 2:
        data = {val: train[train['Education_Status'] == val]['Income'] for val in cluster}
        t_stat, p_val = stats.mannwhitneyu(*data.values())
        print(f"t-statistic: {t_stat}")
        print(f"p-value: {p_val}")
        print()
    else:
        data = {val: train[train['Education_Status'] == val]['Income'] for val in cluster}
        f_stat, p_val = stats.kruskal(*data.values())
        print(f"F-statistic: {f_stat}")
        print(f"p-value: {p_val}")
        print()


In [ ]:
# 각 그룹화된 cluster별로 boxplot
fig, ax = plt.subplots(3, 1, figsize=(8, 12))
sns.boxplot(x='Education_Status', y='Income', data=train[train['Education_Status'].isin(cluster_1)], ax=ax[0])
sns.boxplot(x='Education_Status', y='Income', data=train[train['Education_Status'].isin(cluster_2)], ax=ax[1])
sns.boxplot(x='Education_Status', y='Income', data=train[train['Education_Status'].isin(cluster_3)], ax=ax[2])

# 스케일 고정
for i in range(3):
        ax[i].set_ylim(0, train['Income'].max()+1000)

In [ ]:
# Education_Status를 그룹화하는 함수 정의

def Education_assign_cluster(status):
    if status in cluster_1:
        return 'cluster_1'
    elif status in cluster_2:
        return 'cluster_2'
    elif status in cluster_3:
        return 'cluster_3'
    else:
        return status

def apply_Education(train, test):
    train['Education_Status'] = train['Education_Status'].apply(Education_assign_cluster)
    test['Education_Status'] = test['Education_Status'].apply(Education_assign_cluster)
    return train, test

In [ ]:
# Education_Status 범주화 함수 적용
train, test = apply_Education(train, test)
train['Education_Status'].value_counts()

In [ ]:
# 범주 그룹화 후 Education_Status와 Income의 boxplot
sns.boxplot(y='Education_Status', x='Income', data=train, hue='Education_Status')

## Employment_Status

- 'Full-Time': '전일제'
- 'Children or Armed Forces': '어린이 또는 군인'
- 'Not Working': '무직'
- 'Seeking Full-Time': '전일제 구직 중'
- 'Choice Part-Time': '선택적 알바'
- 'Part-Time (Usually Part-Time)': '알바 (보통 알바)'
- 'Part-Time (Usually Full-Time)': '알바 (보통 전일제)'
- 'Seeking Part-Time': '알바 구직 중'

In [ ]:
train['Employment_Status'].value_counts()

In [ ]:
# Employment_Status와 Income의 boxplot
sns.boxplot(y='Employment_Status', x='Income', data=train, hue='Employment_Status')

### 모순이라 판단한 범주 삭제

- 구직 중인데 근로 소득이 0이 아닌 경우는 모순이라고 판단 -> 확실히 0이라고 판단 후 제거

In [ ]:
# Employment_Status가 Seeking Full-Time인 경우 Income 값 확인
train[train['Employment_Status'] == 'Seeking Full-Time']['Income'].value_counts()

In [ ]:
# Employment_Status가 Seeking Part-Time인 경우 Income 값 확인
train[train['Employment_Status'] == 'Seeking Part-Time']['Income'].value_counts()

In [ ]:
# Employment_Status가 Seeking Full-Time, Seeking Part-Time 이면 Income이 0 -> 미리 처리
def delete_seek_employ(train, test):
    train = train[train['Employment_Status'] != 'Seeking Full-Time']
    train = train[train['Employment_Status'] != 'Seeking Part-Time']
    test = test[test['Employment_Status'] != 'Seeking Full-Time']
    test = test[test['Employment_Status'] != 'Seeking Part-Time']
    return train, test

In [ ]:
# 범주 제거하는 함수 적용
train, test = delete_seek_employ(train, test)
train['Employment_Status'].value_counts()

In [ ]:
# 범주 제거한 뒤 Employment_Status와 Income의 boxplot
sns.boxplot(y='Employment_Status', x='Income', data=train, hue='Employment_Status')

### 범주들 그룹화

- Kruskal-Wallis H 검정 시행 -> Children or Armed Forces를 제외한 범주들 간의 차이가 없음
- 따라서, Children or Armed Forces 값 여부로 범주 변경
- 7개 범주에서 2개 범주로 축소

In [ ]:
# 교육 수준에 따른 소득 데이터 추출
Not_Children_or_Armed_Forces =  ['Full-Time','Choice Part-Time', 'Part-Time (Usually Part-Time)', 'Part-Time (Usually Full-Time)']
data = {val: train[train['Employment_Status'] == val]['Income'] for val in Not_Children_or_Armed_Forces}

# kruskal 수행
f_stat, p_val = stats.kruskal(*data.values())
print(f"F-statistic: {f_stat}")
print(f"p-value: {p_val}")

In [ ]:
# Employment_Status의 범주를 그룹화하는 함수 정의
def Employment_assign_cluster(status):
    if status == 'Children or Armed Forces':
        return 'Children or Armed Forces'
    else:
        return 'Not Children or Armed Forces'
    
def apply_Employment(train, test):
    train['Employment_Status'] = train['Employment_Status'].apply(Employment_assign_cluster)
    test['Employment_Status'] = test['Employment_Status'].apply(Employment_assign_cluster)
    return train, test

In [ ]:
# Employment_Status의 범주를 그룹화하는 함수 적용
train, test = apply_Employment(train, test)
train['Employment_Status'].value_counts()

In [ ]:
# 범주 그룹화 후 Employment_Status와 Income의 boxplot
sns.boxplot(x='Employment_Status', y='Income', data=train, hue='Employment_Status')

## Working_Week (Yearly)

- 연간 몇주동안 일하는지

In [ ]:
train['Working_Week (Yearly)'].value_counts()

In [ ]:
# Working_Week (Yearly)의 histogram
sns.histplot(x='Working_Week (Yearly)', data=train)

In [ ]:
# Working_Week (Yearly)와 Income의 scatterplot
sns.scatterplot(x='Working_Week (Yearly)', y='Income', data=train)

In [ ]:
# Income과 Working_Week (Yearly)의 상관계수
train['Working_Week (Yearly)'].corr(train['Income'])

### 데이터 구간화 진행

기준 : 0 / 1~10 / 11~20 / 21~ 0 / 31~40 / 41~51 / 52
- 범주 7개 생성

In [ ]:
train['Working_Week (Yearly)'] = pd.cut(train['Working_Week (Yearly)'], bins=[-1, 0, 10, 20, 30, 40, 51, 52], labels=[0, 1, 2, 3, 4, 5, 6])
train['Working_Week (Yearly)'].value_counts()

In [ ]:
# 각 범주별로 Income의 boxplot 확인
sns.boxplot(y='Working_Week (Yearly)', x='Income', data=train, hue='Working_Week (Yearly)')
# 범례 제거
plt.legend([],[], frameon=False)

### 범주들 그룹화

- 유사한 분포를 보이는 범주끼리 그룹화 해보자!
- 그 그룹화가 잘 되었는지 확인하기 위해서 통계적 검정 시행

    -> 그룹화 된 범주들 간의 Income 분포 차이가 있는지 확인(정규성을 만족 못한다고 판단하고 비모수 검정으로 진행)
    - 비교하는 범주가 2개면 mann whitney u 검정 시행
    - 비교하는 범주가 3개 이상이면 Kruskal-Wallis H 검정 시행    

- 범주 1, 2, 3 사이에는 차이가 없다
- 범주 4, 5 사이에는 차이가 없다

    -> 0 / 1, 2, 3 / 4, 5 / 6 으로 총 4개의 범주로 그룹화
    - 7개 범주에서 4개 범주로 축소

In [ ]:
# 최종 결정된 그룹들
cluster_1 = [1, 2, 3]
cluster_2 = [4, 5]

In [ ]:
# 그룹 속 범주 간의 Income 차이 검정 -> 그룹 속 범주 간 Income 차이가 없음 
for cluster in [cluster_1, cluster_2]:
    print(f"Working_Week: {cluster}")
    if len(cluster) == 2:
        print('mann whitney u 검정 시행')
        data = {edu: train[train['Working_Week (Yearly)'] == edu]['Income'] for edu in cluster}
        f_stat, p_val = stats.mannwhitneyu(*data.values())
        
        print(f"F-statistic: {f_stat}")
        print(f"p-value: {p_val}")
        print()
    else:
        data = {edu: train[train['Working_Week (Yearly)'] == edu]['Income'] for edu in cluster}
        f_stat, p_val = stats.kruskal(*data.values())
        print('Kruskal-Wallis H 검정 시행')
        print(f"F-statistic: {f_stat}")
        print(f"p-value: {p_val}")
        print()


In [ ]:
# Working_Week의 범주로 그룹화하는 함수 정의
def Working_Week_assign_cluster(label):
    if label < 1:
        return '0'
    elif label < 4:
        return '1_to_30'
    elif label < 6:
        return '31_to_51'
    else:
        return '52'
    
def apply_Working_Week(train, test):
    train['Working_Week (Yearly)'] = train['Working_Week (Yearly)'].apply(Working_Week_assign_cluster)
    test['Working_Week (Yearly)'] = test['Working_Week (Yearly)'].apply(Working_Week_assign_cluster)
    return train, test


In [ ]:
# Working_Week의 범주로 그룹화하는 함수 적용
train, test = apply_Working_Week(train, test)
train['Working_Week (Yearly)'].value_counts()

In [ ]:
# 범주 그룹화 후 Working_Week와 Income의 boxplot
# x값 순서 변경
train['Working_Week (Yearly)'] = train['Working_Week (Yearly)'].astype('category')
sns.boxplot(x='Working_Week (Yearly)', y='Income', data=train, hue='Working_Week (Yearly)')
train['Working_Week (Yearly)'] = train['Working_Week (Yearly)'].astype('object')

## Industry_Status

- 'Social Services': '사회 서비스'
- 'Entertainment': '엔터테인먼트'
- 'Manufacturing (Non-durable)': '제조업 (비내구재)'
- 'Retail': '소매업'
- 'Not in universe or children': '해당 없음 또는 어린이'
- 'Construction': '건설업'
- 'Hospitals': '병원'
- 'Manufacturing (Durable)': '제조업 (내구재)'
- 'Public Administration': '공공 행정'
- 'Education': '교육'
- 'Finance Insurance & Real Estate': '금융 보험 및 부동산'
- 'Transportation': '교통'
- 'Medical (except Hospitals)': '의료 (병원 제외)'
- 'Wholesale': '도매업'
- 'Other professional services': '기타 전문 서비스'
- 'Personal Services (except Private Household)': '개인 서비스 (가사 도우미 제외)'
- 'Business & Repair': '비즈니스 및 수리'
- 'Private Household Services': '가사 도우미 서비스'
- 'Mining': '광업'
- 'Agriculture': '농업'
- 'Utilities & Sanitary': '공공시설 및 위생'
- 'Communications': '통신'
- 'Forestry & Fisheries': '임업 및 어업'
- 'Armed Forces': '군대'

In [ ]:
train['Industry_Status'].value_counts()

In [ ]:
# Industry_Status와 Income의 boxplot
sns.boxplot(y='Industry_Status', x='Income', data=train, hue='Industry_Status')

- 범주가 많음.. 삭제 고려 -> 실험

In [ ]:
# industry_status 제거 함수
def delete_industry(train, test):
    train = train.drop('Industry_Status', axis=1)
    test = test.drop('Industry_Status', axis=1)
    return train, test 

## Occupation_Status

- 'Services': '서비스업'
- 'Admin Support (include Clerical)': '행정 지원 (사무 포함)'
- 'Technicians & Support': '기술자 및 지원'
- 'Sales': '판매'
- 'Unknown': '알 수 없음'
- 'Machine Operators & Inspectors': '기계 조작 및 검사원'
- 'Craft & Repair': '공예 및 수리'
- 'Protective Services': '보호 서비스'
- 'Management': '관리'
- 'Professional': '전문가'
- 'Transportation': '교통'
- 'Handlers/Cleaners': '핸들러/청소원'
- 'Private Household Services': '가사 도우미 서비스'
- 'Farming & Forestry & Fishing': '농업 & 임업 & 어업'
- 'Armed Forces': '군대'

In [ ]:
train['Occupation_Status'].value_counts()

In [ ]:
# Occupation_Status와 Income의 boxplot
sns.boxplot(y='Occupation_Status', x='Income', data=train, hue='Occupation_Status')

- 범주가 많음.. -> 삭제 고려 -> 실험

In [ ]:
# Occupation_Status 제거 함수
def delete_occupation(train, test):
    train = train.drop('Occupation_Status', axis=1)
    test = test.drop('Occupation_Status', axis=1)
    return train, test

## Race

- 인종

- 'White': '백인'
- 'Black': '흑인'
- 'Native American/Aleut/Eskimo': '원주민 아메리칸/알류트인/에스키모'
- 'Asian/Pacific': '아시아/태평양'
- 'Other': '기타'

In [ ]:
train['Race'].value_counts()

In [ ]:
# Race와 Income의 boxplot
sns.boxplot(y='Race', x='Income', data=train, hue='Race')

- Race의 모든 범주 간 Income 분포 차이 검정 -> 귀무가설 기각 못함
- Race 범주 간 Income 차이가 없다 -> Race 컬럼 삭제 고려

In [ ]:
# Race의 모든 범주 간 Income 분포 차이 검정 
data = {val: train[train['Race'] == val]['Income'] for val in train['Race'].unique()}
f_stat, p_val = stats.kruskal(*data.values())
print(f"F-statistic: {f_stat}")
print(f"p-value: {p_val}")

In [ ]:
# industry_status 제거 함수
def delete_race(train, test):
    train = train.drop('Race', axis=1)
    test = test.drop('Race', axis=1)
    return train, test

train, test = delete_race(train, test)

## Hispanic_Origin

- 히스패닉 기원

- ' All other': ' 기타'
- ' Cuban': ' 쿠바인'
- ' Mexican (Mexicano)': ' 멕시코인 (멕시카노)'
- ' Central or South American': ' 중앙 또는 남아메리카인'
- ' Puerto Rican': ' 푸에르토리코인'
- ' Mexican-American': ' 멕시코-아메리칸'
- ' Other Spanish': ' 기타 스페인계'
- ' Do not know': ' 알 수 없음'
- ' Chicano': ' 치카노'
- ' NA': ' 없음'

In [ ]:
train['Hispanic_Origin'].value_counts()

In [ ]:
# Hispanic_Origin과 Income의 boxplot
sns.boxplot(y='Hispanic_Origin', x='Income', data=train, hue='Hispanic_Origin')

### 범주들 그룹화

- All other의 표본 수가 압도적으로 많음
- All other 여부에 따라 그룹화 하고 두 집단 간 Income 차이 확인
- mann whitney u 검정 시행 -> 귀무 가설 기각 -> 두 집단 간 Income 차이가 있다
- 10개 범주에서 2개 범주로 축소

In [ ]:
# 두 집단 분리
Hispanic_Origin_1 = train[train['Hispanic_Origin'] == ' All other']['Income']
Hispanic_Origin_2 = train[train['Hispanic_Origin'] != ' All other']['Income']

# mann whitney u 검정 시행
t_stat, p_val = stats.mannwhitneyu(Hispanic_Origin_1, Hispanic_Origin_2)

print(f"t-statistic: {t_stat}")
print(f"p-value: {p_val}")

In [ ]:
# Hispanic_Origin을 그룹화하는 함수 정의
def Hispanic_Origin_assign_cluster(status):
    if status == ' All other':
        return 'All other'
    else:
        return 'Not_All_other'


def apply_Education(train, test):
    train['Hispanic_Origin'] = train['Hispanic_Origin'].apply(Hispanic_Origin_assign_cluster)
    test['Hispanic_Origin'] = test['Hispanic_Origin'].apply(Hispanic_Origin_assign_cluster)
    return train, test

In [ ]:
# Hispanic_Origin을 그룹화하는 함수 적용
train, test = apply_Education(train, test)
train['Hispanic_Origin'].value_counts()

In [ ]:
# 범주 그룹화 후 Hispanic_Origin와 Income의 boxplot
sns.boxplot(x='Hispanic_Origin', y='Income', data=train, hue = 'Hispanic_Origin')

## Martial_Status

- 'Married': '결혼함'
- 'Separated': '별거중'
- 'Divorced': '이혼함'
- 'Widowed': '사별함'
- 'Single': '미혼'
- 'Married (Spouse Absent)': '결혼함 (배우자 부재)'
- 'Married (Armed Force Spouse)': '결혼함 (군인 배우자)'

In [ ]:
train['Martial_Status'].value_counts()

In [ ]:
# Martial_Status와 Income의 boxplot
sns.boxplot(y='Martial_Status', x='Income', data=train, hue = 'Martial_Status')

### 범주들 그룹화

- 유사한 분포를 보이는 범주끼리 그룹화 해보자!
- 그 그룹화가 잘 되었는지 확인하기 위해서 통계적 검정 시행

    -> 그룹화 된 범주들 간의 Income 분포 차이가 있는지 확인(정규성을 만족 못한다고 판단하고 비모수 검정으로 진행)
    - 비교하는 범주가 2개면 mann whitney u 검정 시행
    - 비교하는 범주가 3개 이상이면 Kruskal-Wallis H 검정 시행    

- 7개 범주에서 3개 범주로 축소

In [ ]:
# 최종 결정된 그룹들
cluster_1 = ['Married', 'Divorced','Married (Armed Force Spouse)']
cluster_2 = ['Separated', 'Widowed','Married (Spouse Absent)']

In [ ]:
# 그룹 속 범주 간의 Income 차이 검정 -> 그룹 속 범주 간 Income 차이가 없음 
for cluster in [cluster_1, cluster_2]:
    print(f"Martial_Status: {cluster}")
    if len(cluster) == 1:
        print('범주 1개라 시행안함')
        continue
    elif len(cluster) == 2:
        data = {edu: train[train['Martial_Status'] == edu]['Income'] for edu in cluster}
        t_stat, p_val = stats.mannwhitneyu(*data.values())
        print(f"t-statistic: {t_stat}")
        print(f"p-value: {p_val}")
        print()
    else:
        data = {edu: train[train['Martial_Status'] == edu]['Income'] for edu in cluster}
        f_stat, p_val = stats.kruskal(*data.values())
        print(f"F-statistic: {f_stat}")
        print(f"p-value: {p_val}")
        print()

In [ ]:
# Martial_Status를 그룹화하는 함수 정의
def Martial_Status_assign_cluster(status):
    if status in cluster_1:
        return 'cluster_1'
    elif status in cluster_2:
        return 'cluster_2'
    else:
        return 'Single'
    
def apply_Martial_Status(train, test):
    train['Martial_Status'] = train['Martial_Status'].apply(Martial_Status_assign_cluster)
    test['Martial_Status'] = test['Martial_Status'].apply(Martial_Status_assign_cluster)
    return train, test

In [ ]:
# Martial_Status 범주화 함수 적용
train, test = apply_Martial_Status(train, test)
train['Martial_Status'].value_counts()

In [ ]:
# 범주 그룹화 후 Martial_Status와 Income의 boxplot
sns.boxplot(x='Martial_Status', y='Income', data=train, hue='Martial_Status')

## Household_Status

- 'Householder': '가구주'
- 'Nonfamily householder': '비가족 가구주'
- 'Child 18+ never marr Not in a subfamily': '18세 이상 미혼, 부가족에 속하지 않음'
- 'Spouse of householder': '가구주의 배우자'
- 'Other Relative 18+ ever married Responsible Person of subfamily': '18세 이상 결혼 경험 있음, 부가족의 책임자인 기타 친족'
- 'Child <18 never marr not in subfamily': '18세 미만 미혼, 부가족에 속하지 않음'
- 'Other Relative <18 never married child of subfamily Responsible Person': '18세 미만 미혼, 부가족의 책임자인 기타 친족'
- 'Child 18+ ever married Responsible Person of subfamily': '18세 이상 결혼 경험 있음, 부가족의 책임자'
- 'Secondary individual': '2차 개인'
- 'Grandchild 18+ never marr not in subfamily': '18세 이상 미혼, 부가족에 속하지 않는 손자/손녀'
- 'Other Rel 18+ ever marr not in subfamily': '18세 이상 결혼 경험 있음, 부가족에 속하지 않는 기타 친족'
- 'Responsible Person of unrelated subfamily': '비친족 부가족의 책임자'
- 'Grandchild <18 never married child of subfamily Responsible Person': '18세 미만 미혼, 부가족의 책임자인 손자/손녀'
- 'Child 18+ ever marr Not in a subfamily': '18세 이상 결혼 경험 있음, 부가족에 속하지 않음'
- 'Other Rel 18+ never marr not in subfamily': '18세 이상 미혼, 부가족에 속하지 않는 기타 친족'
- 'Child 18+ never married Responsible Person of subfamily': '18세 이상 미혼, 부가족의 책임자'
- 'Other Rel <18 never marr not in subfamily': '18세 미만 미혼, 부가족에 속하지 않는 기타 친족'
- 'Child under 18 of Responsible Person of unrelated subfamily': '비친족 부가족의 책임자인 18세 미만 아동'
- 'Grandchild <18 never marr not in subfamily': '18세 미만 미혼, 부가족에 속하지 않는 손자/손녀'
- 'In group quarters': '집단 숙소에 속함'
- 'Other Relative 18+ spouse of subfamily Responsible Person': '18세 이상, 부가족의 책임자의 배우자인 기타 친족'
- 'Other Relative 18+ never married Responsible Person of subfamily': '18세 이상 미혼, 부가족의 책임자인 기타 친족'
- 'Child <18 never married Responsible Person of subfamily': '18세 미만 미혼, 부가족의 책임자'
- 'Grandchild 18+ spouse of subfamily Responsible Person': '18세 이상, 부가족의 책임자의 배우자인 손자/손녀'
- 'Child 18+ spouse of subfamily Responsible Person': '18세 이상, 부가족의 책임자의 배우자'
- 'Child <18 ever marr not in subfamily': '18세 미만 결혼 경험 있음, 부가족에 속하지 않음'
- 'Spouse of Responsible Person of unrelated subfamily': '비친족 부가족의 책임자의 배우자'
- 'Grandchild 18+ ever marr not in subfamily': '18세 이상 결혼 경험 있음, 부가족에 속하지 않는 손자/손녀'
- 'Grandchild 18+ ever married Responsible Person of subfamily': '18세 이상 결혼 경험 있음, 부가족의 책임자인 손자/손녀'
- 'Child <18 ever married Responsible Person of subfamily': '18세 미만 결혼 경험 있음, 부가족의 책임자'
- 'Other Relative <18 ever married Responsible Person of subfamily': '18세 미만 결혼 경험 있음, 부가족의 책임자인 기타 친족'

In [ ]:
train['Household_Status'].nunique()

In [ ]:
train['Household_Status'].value_counts()

In [ ]:
# Household_Status와 Income의 boxplot
fig, ax = plt.subplots(1, 1, figsize=(12, 12))
sns.boxplot(y='Household_Status', x='Income', data=train, hue='Household_Status')

31개의 범주.. 너무 많다.. -> drop 후 household_summary 활용 고민

In [ ]:
# Household_Status 제거
train.drop('Household_Status', axis=1, inplace=True)
test.drop('Household_Status', axis=1, inplace=True)

## Household_Summary

- 'Householder': '가구주'
- 'Child 18 or older': '18세 이상의 자녀'
- 'Spouse of householder': '가구주의 배우자'
- 'Other relative of householder': '가구주의 기타 친족'
- 'Child under 18 never married': '18세 미만 미혼 자녀'
- 'Nonrelative of householder': '가구주와 친족 관계 없음'
- 'Group Quarters- Secondary individual': '집단 숙소- 2차 개인'
- 'Child under 18 ever married': '18세 미만 결혼 경험 있는 자녀'

In [ ]:
train['Household_Summary'].value_counts()

In [ ]:
# Household_Summary와 Income의 boxplot
sns.boxplot(y='Household_Summary', x='Income', data=train, hue='Household_Summary')

### 범주들 그룹화

- 유사한 분포를 보이는 범주끼리 그룹화 해보자!
- 그 그룹화가 잘 되었는지 확인하기 위해서 통계적 검정 시행

    -> 그룹화 된 범주들 간의 Income 분포 차이가 있는지 확인(정규성을 만족 못한다고 판단하고 비모수 검정으로 진행)
    - 비교하는 범주가 2개면 mann whitney u 검정 시행
    - 비교하는 범주가 3개 이상이면 Kruskal-Wallis H 검정 시행    
- 8개 범주에서 5개 범주로 축소

In [ ]:
# 최종 결정된 그룹들
cluster_1 = ['Householder', 'Child under 18 ever married']
cluster_2 = ['Other relative of householder','Nonrelative of householder', 'Group Quarters- Secondary individual']

In [ ]:
# 그룹 속 범주 간의 Income 차이 검정 -> 그룹 속 범주 간 Income 차이가 없음 
for cluster in [cluster_1, cluster_2]:
    print(f"Household_Summary: {cluster}")
    if len(cluster) == 1:
        print('범주 1개라 시행안함')
        continue
    elif len(cluster) == 2:
        data = {edu: train[train['Household_Summary'] == edu]['Income'] for edu in cluster}
        t_stat, p_val = stats.mannwhitneyu(*data.values())
        print(f"t-statistic: {t_stat}")
        print(f"p-value: {p_val}")
        print()
    else:
        data = {edu: train[train['Household_Summary'] == edu]['Income'] for edu in cluster}
        f_stat, p_val = stats.kruskal(*data.values())
        print(f"F-statistic: {f_stat}")
        print(f"p-value: {p_val}")
        print()

In [ ]:
# Household_Summary를 그룹화하는 함수 정의
def Household_Summary_assign_cluster(status):
    if status in cluster_1:
        return 'cluster_1'
    elif status in cluster_2:
        return 'cluster_2'
    else:
        return status

def apply_Household_Summary(train, test):
    train['Household_Summary'] = train['Household_Summary'].apply(Household_Summary_assign_cluster)
    test['Household_Summary'] = test['Household_Summary'].apply(Household_Summary_assign_cluster)
    return train, test

In [ ]:
# Household_Summary 범주화 함수 적용
train, test = apply_Household_Summary(train, test)
train['Household_Summary'].value_counts()

In [ ]:
# 범주 그룹화 후 Household_Summary와 Income의 boxplot
sns.boxplot(y='Household_Summary', x='Income', data=train, hue = 'Household_Summary')

## Citizenship

- 'Native': '원주민'
- 'Foreign-born (Non-US Citizen)': '외국 출생 (미국 시민권 없음)'
- 'Native (Born in Puerto Rico or US Outlying)': '원주민 (푸에르토리코 또는 미국령 외곽지역 출생)'
- 'Foreign-born (Naturalized US Citizen)': '외국 출생 (귀화 미국 시민)'
- 'Native (Born Abroad)': '원주민 (해외 출생)'

In [ ]:
train['Citizenship'].value_counts()

In [ ]:
# Citizenship와 Income의 boxplot
sns.boxplot(y='Citizenship', x='Income', data=train, hue='Citizenship')

### 범주들 그룹화

- 유사한 분포를 보이는 범주끼리 그룹화 해보자!
- 그 그룹화가 잘 되었는지 확인하기 위해서 통계적 검정 시행

    -> 그룹화 된 범주들 간의 Income 분포 차이가 있는지 확인(정규성을 만족 못한다고 판단하고 비모수 검정으로 진행)
    - 비교하는 범주가 2개면 mann whitney u 검정 시행
    - 비교하는 범주가 3개 이상이면 Kruskal-Wallis H 검정 시행    
- 5개 범주에서 2개 범주로 축소

In [ ]:
# 최종 결정된 그룹들
cluster_1 = ['Native', 'Foreign-born (Naturalized US Citizen)']
cluster_2 = ['Foreign-born (Non-US Citizen)', 'Native (Born Abroad)', 'Native (Born in Puerto Rico or US Outlying)']

In [ ]:
# 그룹 속 범주 간의 Income 차이 검정 -> 그룹 속 범주 간 Income 차이가 없음 
for cluster in [cluster_1, cluster_2]:
    print(f"Citizenship: {cluster}")
    if len(cluster) == 1:
        print('범주 1개라 시행안함')
        continue
    elif len(cluster) == 2:
        data = {edu: train[train['Citizenship'] == edu]['Income'] for edu in cluster}
        t_stat, p_val = stats.mannwhitneyu(*data.values())
        print(f"t-statistic: {t_stat}")
        print(f"p-value: {p_val}")
        print()
    else:
        data = {edu: train[train['Citizenship'] == edu]['Income'] for edu in cluster}
        f_stat, p_val = stats.kruskal(*data.values())
        print(f"F-statistic: {f_stat}")
        print(f"p-value: {p_val}")
        print()

In [ ]:
# Citizenship를 그룹화하는 함수 정의
def Citizenship_assign_cluster(status):
    if status in cluster_1:
        return 'cluster_1'
    elif status in cluster_2:
        return 'cluster_2'
    else:
        return status
        
def apply_Citizenship(train, test):
    train['Citizenship'] = train['Citizenship'].apply(Citizenship_assign_cluster)
    test['Citizenship'] = test['Citizenship'].apply(Citizenship_assign_cluster)
    return train, test

In [ ]:
# Citizenship을 그룹화하는 함수 적용
train, test = apply_Citizenship(train, test)
train['Citizenship'].value_counts()

In [ ]:
# 범주 그룹화 후 Citizenship와 Income의 boxplot
sns.boxplot(x='Citizenship', y='Income', data=train, hue='Citizenship')

## Birth_Country

In [ ]:
train['Birth_Country'].value_counts()

### 범주들 그룹화

- US인 행과 아닌 행으로 범주화

- 독립표본 t test 시행 시도
- 정규성 검정, 등분산 검정 시도 -> 기각 -> 비모수 방법인 mann whitney u 사용
- 귀무가설 기각(p value < 0.05) -> 두 집단은 Income에 대해 유의미한 차이가 있다.

In [ ]:
birth_US = train[train['Birth_Country'] == 'US']['Income']
birth_not_US = train[train['Birth_Country'] != 'US']['Income']

# 정규성 검정 -> p < 0.05 이므로 정규성 만족 X
print('M 집단 정규성 검정 결과 :', stats.shapiro(birth_US))
print('F 집단 정규성 검정 결과 :', stats.shapiro(birth_not_US))
print()

# 등분산 검정 -> p < 0.05 이므로 등분산 만족 X
print('M, F 집단 간 등분산성 검정 결과 :',stats.levene(birth_US, birth_not_US))
print()

# 독립표본 t 검정 -> p < 0.05 이므로 두 집단의 소득 차이가 유의미함
print('mann whitney u 검정 결과 :',stats.ttest_ind(birth_US, birth_not_US))
# 비모수 검정 -> p < 0.05 이므로 두 집단의 소득 차이가 유의미함
print('mann whitney u 검정 결과 :',stats.mannwhitneyu(birth_US, birth_not_US))

In [ ]:
# Birth_Country의 범주를 그룹화하는 함수 정의
def Birth_Country_assign_cluster(status):
    if status == 'US':
        return 'US'
    else:
        return 'Not US'
    
def apply_Birth_Country(train, test):
    train['Birth_Country'] = train['Birth_Country'].apply(Birth_Country_assign_cluster)
    test['Birth_Country'] = test['Birth_Country'].apply(Birth_Country_assign_cluster)
    return train, test


In [ ]:
# Birth_Country의 범주를 그룹화하는 함수 적용
train, test = apply_Birth_Country(train, test)
train['Birth_Country'].value_counts()

In [ ]:
# 범주 그룹화 후 Birth_Country와 Income의 boxplot
sns.boxplot(x='Birth_Country', y='Income', data=train, hue='Birth_Country')

## Birth_Country (Father)

In [ ]:
train['Birth_Country (Father)'].value_counts()

### 범주들 그룹화

- US인 행과 아닌 행으로 범주화

- 독립표본 t test 시행 시도
- 정규성 검정, 등분산 검정 시도 -> 기각 -> 비모수 방법인 mann whitney u 사용
- 귀무가설 기각(p value < 0.05) -> 두 집단은 Income에 대해 유의미한 차이가 있다.

In [ ]:
birth_US = train[train['Birth_Country (Father)'] == 'US']['Income']
birth_not_US = train[train['Birth_Country (Father)'] != 'US']['Income']

# 정규성 검정 -> p < 0.05 이므로 정규성 만족 X
print('M 집단 정규성 검정 결과 :', stats.shapiro(birth_US))
print('F 집단 정규성 검정 결과 :', stats.shapiro(birth_not_US))
print()

# 등분산 검정 -> p < 0.05 이므로 등분산 만족 X
print('M, F 집단 간 등분산성 검정 결과 :',stats.levene(birth_US, birth_not_US))
print()

# 독립표본 t 검정 -> p < 0.05 이므로 두 집단의 소득 차이가 유의미함
print('mann whitney u 검정 결과 :',stats.ttest_ind(birth_US, birth_not_US))
# 비모수 검정 -> p < 0.05 이므로 두 집단의 소득 차이가 유의미함
print('mann whitney u 검정 결과 :',stats.mannwhitneyu(birth_US, birth_not_US))

In [ ]:
# Birth_Country (Father)의 범주를 그룹화하는 함수 정의
def Birth_Country_F_assign_cluster(status):
    if status == 'US':
        return 'US'
    else:
        return 'Not US'
    
def apply_Birth_F_Country(train, test):
    train['Birth_Country (Father)'] = train['Birth_Country (Father)'].apply(Birth_Country_F_assign_cluster)
    test['Birth_Country (Father)'] = test['Birth_Country (Father)'].apply(Birth_Country_F_assign_cluster)
    return train, test

In [ ]:
# Birth_Country의 범주를 그룹화하는 함수 적용
train, test = apply_Birth_F_Country(train, test)
train['Birth_Country (Father)'].value_counts()

In [ ]:
# 범주 그룹화 후 Birth_Country와 Income의 boxplot
sns.boxplot(x='Birth_Country (Father)', y='Income', data=train, hue='Birth_Country (Father)')

## Birth_Country (Mother)

In [ ]:
train['Birth_Country (Mother)'].value_counts()

### 범주들 그룹화

- US인 행과 아닌 행으로 범주화

- 독립표본 t test 시행 시도
- 정규성 검정, 등분산 검정 시도 -> 기각 -> 비모수 방법인 mann whitney u 사용
- 귀무가설 기각(p value < 0.05) -> 두 집단은 Income에 대해 유의미한 차이가 있다.

In [ ]:
birth_US = train[train['Birth_Country (Mother)'] == 'US']['Income']
birth_not_US = train[train['Birth_Country (Mother)'] != 'US']['Income']

# 정규성 검정 -> p < 0.05 이므로 정규성 만족 X
print('M 집단 정규성 검정 결과 :', stats.shapiro(birth_US))
print('F 집단 정규성 검정 결과 :', stats.shapiro(birth_not_US))
print()

# 등분산 검정 -> p < 0.05 이므로 등분산 만족 X
print('M, F 집단 간 등분산성 검정 결과 :',stats.levene(birth_US, birth_not_US))
print()

# 독립표본 t 검정 -> p < 0.05 이므로 두 집단의 소득 차이가 유의미함
print('mann whitney u 검정 결과 :',stats.ttest_ind(birth_US, birth_not_US))
# 비모수 검정 -> p < 0.05 이므로 두 집단의 소득 차이가 유의미함
print('mann whitney u 검정 결과 :',stats.mannwhitneyu(birth_US, birth_not_US))

In [ ]:
# Birth_Country (Mother)의 범주를 그룹화하는 함수 정의
def Birth_Country_M_assign_cluster(status):
    if status == 'US':
        return 'US'
    else:
        return 'Not US'
    
def apply_Birth_M_Country(train, test):
    train['Birth_Country (Mother)'] = train['Birth_Country (Mother)'].apply(Birth_Country_M_assign_cluster)
    test['Birth_Country (Mother)'] = test['Birth_Country (Mother)'].apply(Birth_Country_M_assign_cluster)
    return train, test

In [ ]:
# Birth_Country의 범주를 그룹화하는 함수 적용
train, test = apply_Birth_M_Country(train, test)
train['Birth_Country (Mother)'].value_counts()

In [ ]:
# 범주 그룹화 후 Birth_Country와 Income의 boxplot
sns.boxplot(x='Birth_Country (Mother)', y='Income', data=train, hue='Birth_Country (Mother)')

## Tax_Status

- 'Nonfiler': '신고하지 않음'
- 'Single': '단독'
- 'Married Filling Jointly both under 65 (MFJ)': '결혼, 65세 미만 부부 공동 신고 (MFJ)'
- 'Head of Household (HOH)': '가장 (HOH)'
- 'Married Filling Jointly one over 65 & one under 65 (MFJ)': '결혼, 한 명은 65세 이상 & 한 명은 65세 미만 부부 공동 신고 (MFJ)'
- 'Married Filling Jointly both over 65 (MFJ)': '결혼, 65세 이상 부부 공동 신고 (MFJ)'

In [ ]:
train['Tax_Status'].value_counts()

In [ ]:
# Tax_Status와 Income의 boxplot
sns.boxplot(y='Tax_Status', x='Income', data=train, hue='Tax_Status')

### 범주들 그룹화

- 유사한 분포를 보이는 범주끼리 그룹화 해보자!
- 그 그룹화가 잘 되었는지 확인하기 위해서 통계적 검정 시행

    -> 그룹화 된 범주들 간의 Income 분포 차이가 있는지 확인(정규성을 만족 못한다고 판단하고 비모수 검정으로 진행)
    - 비교하는 범주가 2개면 mann whitney u 검정 시행
    - 비교하는 범주가 3개 이상이면 Kruskal-Wallis H 검정 시행    
- 6개 범주에서 5개 범주로 축소

In [ ]:
# 최종 결정된 그룹
cluster_1 = ['Married Filling Jointly one over 65 & one under 65 (MFJ)','Married Filling Jointly both over 65 (MFJ)']

In [ ]:
# 그룹 속 범주 간의 Income 차이 검정 -> 그룹 속 범주 간 Income 차이가 없음 

for cluster in [cluster_1]:
    print(f"Tax_Status: {cluster}")
    if len(cluster) == 1:
        print('범주 1개라 시행안함')
        continue
    elif len(cluster) == 2:
        data = {val: train[train['Tax_Status'] == val]['Income'] for val in cluster}
        t_stat, p_val = stats.mannwhitneyu(*data.values())
        print(f"t-statistic: {t_stat}")
        print(f"p-value: {p_val}")
        print()
    else:
        data = {val: train[train['Tax_Status'] == val]['Income'] for val in cluster}
        f_stat, p_val = stats.kruskal(*data.values())
        print(f"F-statistic: {f_stat}")
        print(f"p-value: {p_val}")
        print()

In [ ]:
# Tax_Status를 그룹화하는 함수 정의
def Tax_Status_assign_cluster(status):
    if status in cluster_1:
        return 'cluster_1'
    else:
        return status
    
def apply_Tax_Status(train, test):
    train['Tax_Status'] = train['Tax_Status'].apply(Tax_Status_assign_cluster)
    test['Tax_Status'] = test['Tax_Status'].apply(Tax_Status_assign_cluster)
    return train, test

In [ ]:
# Tax_Status 범주화 함수 적용
train, test = apply_Tax_Status(train, test)
train['Tax_Status'].value_counts()

In [ ]:
# 범주 그룹화 후 Tax_Status와 Income의 boxplot
sns.boxplot(y='Tax_Status', x='Income', data=train, hue='Tax_Status')

## Gains

In [ ]:
train['Gains'].value_counts()

### 이상치 확인 및 처리

In [ ]:
# Gains의 boxplot
sns.boxplot(train['Gains'])

In [ ]:
train[train['Gains'] > 90000]['Gains'].value_counts()

In [ ]:
# Gains의 이상치 제거
train = train[train['Gains'] < 99999]
# 이상치 제거 후 Gains의 boxplot
sns.boxplot(train['Gains'])

## Losses

In [ ]:
train['Losses'].value_counts()

In [ ]:
# Losses의 boxplot
sns.boxplot(train['Losses'])

### Gains과 Losses 간 관계

In [ ]:
train[(train['Gains'] == 0) & (train['Losses'] == 0)][['Gains', 'Losses']]

In [ ]:
train[(train['Gains'] == 0) & (train['Losses'] != 0)][['Gains', 'Losses']]

In [ ]:
train[(train['Gains'] != 0) & (train['Losses'] == 0)][['Gains', 'Losses']]

In [ ]:
train[(train['Gains'] != 0) & (train['Losses'] != 0)][['Gains', 'Losses']]

- Gains과 Losses 둘 다 값을 가지는 경우 없음 -> Gains과 Losses를 이용해 총 이득을 구할 수 있음
- Gains - Losses

### Gains, Losses 통합 -> total 변수 생성

In [ ]:
train['total'] = train['Gains'] - train['Losses']
train = train.drop('Gains', axis=1)
train = train.drop('Losses', axis=1)

test['total'] = test['Gains'] - test['Losses']
test = test.drop('Gains', axis=1)
test = test.drop('Losses', axis=1)

In [ ]:
# total의 boxplot
sns.boxplot(train['total'])

In [ ]:
# Gains와 Losses의 통합 후 Income과의 scatterplot
sns.scatterplot(x='total', y='Income', data=train)

In [ ]:
sns.scatterplot(x='total', y='Income', data=train[train['total'] != 0])

In [ ]:
# total과 income의 상관계수
train['total'].corr(train['Income'])

### total 변수의 그룹화

- total 값의 0 여부에 따라 그룹화

- 독립표본 t test 시행 시도
- 정규성 검정, 등분산 검정 시도 -> 기각 -> 비모수 방법인 mann whitney u 사용
- 귀무가설 기각(p value < 0.05) -> 두 집단은 Income에 대해 유의미한 차이가 있다.

In [ ]:
# Employment_Status의 범주를 그룹화하는 함수 정의
def total_assign_cluster(val):
    if val == 0:
        return 'zero'
    else:
        return 'non_zero'
    
def apply_total(train, test):
    train['total'] = train['total'].apply(total_assign_cluster)
    test['total'] = test['total'].apply(total_assign_cluster)
    return train, test

In [ ]:
train, test = apply_total(train, test)

In [ ]:
total_zero = train[train['total'] == 'zero']['Income']
total_non_zero = train[train['total'] == 'non_zero']['Income']


# 정규성 검정 -> p < 0.05 이므로 정규성 만족 X
print('M 집단 정규성 검정 결과 :', stats.shapiro(total_zero))
print('F 집단 정규성 검정 결과 :', stats.shapiro(total_non_zero))
print()

# 등분산 검정 -> p > 0.05 이므로 등분산 만족
print('M, F 집단 간 등분산성 검정 결과 :',stats.levene(total_zero, total_non_zero))
print()

# 독립표본 t test 시행 -> p < 0.05 이므로 두 집단의 소득 차이가 유의미함
print('t 검정 결과 :', stats.ttest_ind(total_zero, total_non_zero))

# 비모수 검정 -> p < 0.05 이므로 두 집단의 소득 차이가 유의미함
print('mann whitney u 검정 결과 :',stats.mannwhitneyu(total_zero, total_non_zero))

In [ ]:
# 범주화 후 total과 income의 boxplot
sns.boxplot(x='total', y='Income', data=train, hue = 'total')

## Dividends

In [ ]:
train['Dividends'].value_counts()

In [ ]:
sns.scatterplot(x='Dividends', y='Income', data=train)

In [ ]:
sns.scatterplot(x='Dividends', y='Income', data=train[train['Dividends'] != 0])

### dividends 변수의 그룹화

In [ ]:
def dividends_assign_cluster(val):
    if val == 0:
        return 'zero'
    else:
        return 'non_zero'
    
def apply_dividends(train, test):
    train['Dividends'] = train['Dividends'].apply(dividends_assign_cluster)
    test['Dividends'] = test['Dividends'].apply(dividends_assign_cluster)
    return train, test

In [ ]:
train, test = apply_dividends(train, test)
train['Dividends'].value_counts()

In [ ]:
Dividends_zero = train[train['Dividends'] == 'zero']['Income']
Dividends_non_zero = train[train['Dividends'] == 'non_zero']['Income']

t_stat, p_val = stats.mannwhitneyu(Dividends_zero, Dividends_non_zero)

print(f"t-statistic: {t_stat}")
print(f"p-value: {p_val}")

In [ ]:
total_zero = train[train['total'] == 'zero']['Income']
total_non_zero = train[train['total'] == 'non_zero']['Income']


# 정규성 검정 -> p < 0.05 이므로 정규성 만족 X
print('M 집단 정규성 검정 결과 :', stats.shapiro(total_zero))
print('F 집단 정규성 검정 결과 :', stats.shapiro(total_non_zero))
print()

# 등분산 검정 -> p > 0.05 이므로 등분산 만족
print('M, F 집단 간 등분산성 검정 결과 :',stats.levene(total_zero, total_non_zero))
print()

# 독립표본 t test 시행 -> p < 0.05 이므로 두 집단의 소득 차이가 유의미함
print('t 검정 결과 :', stats.ttest_ind(total_zero, total_non_zero))

# 비모수 검정 -> p < 0.05 이므로 두 집단의 소득 차이가 유의미함
print('mann whitney u 검정 결과 :',stats.mannwhitneyu(total_zero, total_non_zero))

In [ ]:
sns.boxplot(x='Dividends', y='Income', data=train, hue = 'Dividends')

## Income_Status

- 'Unknown': '알 수 없음'
- 'Under Median': '중앙값 이하'
- 'Over Median': '중앙값 이상'

In [ ]:
train['Income_Status'].value_counts()

In [ ]:
sns.boxplot(x='Income_Status', y='Income', data=train, hue='Income_Status')

- Income_Status의 모든 범주 간 Income 분포 차이 검정 -> 귀무가설 기각 못함
- Income_Status 범주 간 Income 차이가 없다 -> Income_Status 컬럼 삭제 고려

In [ ]:
# ANOVA 수행
import scipy.stats as stats

# 교육 수준에 따른 소득 데이터 추출
Income_Status = train['Income_Status'].unique()
data = {val: train[train['Income_Status'] == val]['Income'] for val in Income_Status}

# ANOVA 수행
f_stat, p_val = stats.kruskal(*data.values())

print(f"F-statistic: {f_stat}")
print(f"p-value: {p_val}")


In [ ]:
# Income_Status 변수 삭제 함수
def delete_Income_Status(train, test):
    train = train.drop('Income_Status', axis=1)
    test = test.drop('Income_Status', axis=1)
    return train, test
train, test = delete_Income_Status(train, test)

# 범주형 변수 인코딩

In [ ]:
# 각 범주형 변수에 대해서 train 범주랑 test 범주가 같은지 확인하는 코드

for col in train.columns[1:]:
    if (train[col].dtype == 'object') or (train[col].dtype == 'category'):
        if set(train[col].unique()) == set(test[col].unique()):
            print(f'{col}의 train, test 범주가 같음')
        else:
            print(f'{col}의 train, test 범주가 다름')
            print(train[col].nunique())
            print(test[col].nunique())
            # 없는 범주 출력
            print(set(train[col].unique()) - set(test[col].unique()))
            print(set(test[col].unique()) - set(train[col].unique()))
        print('-----------------')

## 라벨인코딩

In [ ]:
from sklearn.preprocessing import LabelEncoder

# 범주형 변수를 숫자로 변환
encoder = LabelEncoder()

for col in train.columns[1:]:
    if (train[col].dtype == 'object') or (train[col].dtype == 'category'):
        train[col] = encoder.fit_transform(train[col])
        test[col] = encoder.transform(test[col])
train

### 함수 정의

In [ ]:
# 라벨 인코딩 함수 정의
from sklearn.preprocessing import LabelEncoder
def apply_label_encoding(train, test):
    encoder = LabelEncoder()
    for col in train.columns[1:]:
        if (train[col].dtype == 'object') or (train[col].dtype == 'category'):
            train[col] = encoder.fit_transform(train[col])
            test[col] = encoder.transform(test[col])
    return train, test

## 원핫인코딩

In [ ]:
col_list = []
for col in train.columns[1:]:
    if (train[col].dtype == 'object') or (train[col].dtype == 'category'):
        col_list.append(col)

In [ ]:
from sklearn.preprocessing import OneHotEncoder
for feature in col_list:
    ohe = OneHotEncoder(sparse_output=False)

    # train 데이터 onehot encoding
    
    temp_train = ohe.fit_transform(pd.DataFrame(train[feature]))
    feature_names = ohe.get_feature_names_out(input_features=[feature])
    temp_train = pd.DataFrame(temp_train, columns=feature_names)
    train.reset_index(drop=True, inplace=True)
    train = pd.concat([train, temp_train], axis=1)
    train = train.drop(feature, axis=1)

    # test 데이터 onehot encoding
    temp_test = ohe.transform(pd.DataFrame(test[feature]))
    feature_names = ohe.get_feature_names_out(input_features=[feature])
    temp_test = pd.DataFrame(temp_test, columns=feature_names)
    test.reset_index(drop=True, inplace=True)
    test = pd.concat([test, temp_test], axis=1)
    test = test.drop(feature, axis=1)

### 함수 정의

In [ ]:
from sklearn.preprocessing import OneHotEncoder
# 원핫 인코딩 함수 정의
def apply_one_hot_encoding(train, test):
    col_list = []
    for col in train.columns[1:]:
        if (train[col].dtype == 'object') or (train[col].dtype == 'category'):
            col_list.append(col)

    for feature in col_list:
        ohe = OneHotEncoder(sparse_output=False)

        # train 데이터 onehot encoding
        
        temp_train = ohe.fit_transform(pd.DataFrame(train[feature]))
        feature_names = ohe.get_feature_names_out(input_features=[feature])
        temp_train = pd.DataFrame(temp_train, columns=feature_names)
        train.reset_index(drop=True, inplace=True)
        train = pd.concat([train, temp_train], axis=1)
        train = train.drop(feature, axis=1)

        # test 데이터 onehot encoding
        temp_test = ohe.transform(pd.DataFrame(test[feature]))
        feature_names = ohe.get_feature_names_out(input_features=[feature])
        temp_test = pd.DataFrame(temp_test, columns=feature_names)
        test.reset_index(drop=True, inplace=True)
        test = pd.concat([test, temp_test], axis=1)
        test = test.drop(feature, axis=1)
    return train, test

## 타겟인코딩

In [ ]:
from sklearn.preprocessing import TargetEncoder
col_list = []
for col in train.columns[1:]:
    if (train[col].dtype == 'object') or (train[col].dtype == 'category'):
        col_list.append(col)

for i in col_list:
    le = TargetEncoder()
    train[i] = le.fit_transform(train[i].values.reshape(-1, 1), train['Income'].values.reshape(-1, 1))
    test[i] = le.transform(test[i].values.reshape(-1, 1))

### 함수 정의

In [ ]:
from sklearn.preprocessing import TargetEncoder
# 타겟 인코딩 함수 정의
def apply_target_encoding(train, test):
    col_list = []
    for col in train.columns[1:]:
        if (train[col].dtype == 'object') or (train[col].dtype == 'category'):
            col_list.append(col)

    for i in col_list:
        le = TargetEncoder()
        train[i] = le.fit_transform(train[i].values.reshape(-1, 1), train['Income'].values.reshape(-1, 1))
        test[i] = le.transform(test[i].values.reshape(-1, 1))
    return train, test

# 실험

## 데이터 다시 불러오기

In [ ]:
train = pd.read_csv('../data/raw/train.csv')
test = pd.read_csv('../data/raw/test.csv')

## 확정 전처리 및 feature engineering 적용하는 함수 생성 

In [ ]:
def preprocess(train, test):
    # 확실한 income 0 처리 -> 모델 구축에 사용한 데이터에서 제외
    train = train[train['Education_Status'] != 'Children']
    train = train[train['Employment_Status'] != 'Not Working']
    train = train[train['Industry_Status'] != 'Not in universe or children']
    train = train[train['Occupation_Status'] != 'Unknown']
    train = train[train['Age'] > 14]

    test = test[test['Education_Status'] != 'Children']
    test = test[test['Employment_Status'] != 'Not Working']
    test = test[test['Industry_Status'] != 'Not in universe or children']
    test = test[test['Occupation_Status'] != 'Unknown']
    test = test[test['Age'] > 14]
    # Gains 이상치 처리
    train = train[train['Gains'] < 99999]
    # Household_Status 제거
    train.drop('Household_Status', axis=1, inplace=True)
    test.drop('Household_Status', axis=1, inplace=True)
    # Gains, Losses 통합
    train['total'] = train['Gains'] - train['Losses']
    train = train.drop('Gains', axis=1)
    train = train.drop('Losses', axis=1)

    test['total'] = test['Gains'] - test['Losses']
    test = test.drop('Gains', axis=1)
    test = test.drop('Losses', axis=1)
    
    return train, test

In [ ]:
train, test = preprocess(train, test)
train.shape, test.shape

## feature engineering 방법 함수 정의

In [ ]:
# Education_Status 범주 그룹화
def Education_assign_cluster(status):
    # 최종 결정된 그룹들
    cluster_1 = ['Kindergarten', 'Elementary (1-4)', 'Elementary (5-6)']
    cluster_2 = ['Middle (7-8)','High Freshman', 'High Sophomore','High Junior', 'High Senior']
    cluster_3 = ['Professional degree', 'Masters degree']
    if status in cluster_1:
        return 'cluster_1'
    elif status in cluster_2:
        return 'cluster_2'
    elif status in cluster_3:
        return 'cluster_3'
    else:
        return status


def apply_Education(train, test):
    train['Education_Status'] = train['Education_Status'].apply(Education_assign_cluster)
    test['Education_Status'] = test['Education_Status'].apply(Education_assign_cluster)
    return train, test

In [ ]:
# Employment에서 모순이라 판단한 범주 처리 -> 모델 구축에 사용한 데이터에서 제외
def delete_seek_employ(train, test):
    train = train[train['Employment_Status'] != 'Seeking Full-Time']
    train = train[train['Employment_Status'] != 'Seeking Part-Time']
    test = test[test['Employment_Status'] != 'Seeking Full-Time']
    test = test[test['Employment_Status'] != 'Seeking Part-Time']
    return train, test

In [ ]:
# Employment 범주 그룹화

# Employment_Status의 범주를 그룹화하는 함수 정의
def Employment_assign_cluster(status):
    if status == 'Children or Armed Forces':
        return 'Children or Armed Forces'
    else:
        return 'Not Children or Armed Forces'
    
def apply_Employment(train, test):
    train, test = delete_seek_employ(train, test)
    train['Employment_Status'] = train['Employment_Status'].apply(Employment_assign_cluster)
    test['Employment_Status'] = test['Employment_Status'].apply(Employment_assign_cluster)
    return train, test

In [ ]:
# Working_Week 데이터 구간화

def Working_Week_binning(train, test):
    train['Working_Week (Yearly)'] = pd.cut(train['Working_Week (Yearly)'], bins=[-1, 0, 10, 20, 30, 40, 51, 52], labels=[0, 1, 2, 3, 4, 5, 6])
    test['Working_Week (Yearly)'] = pd.cut(test['Working_Week (Yearly)'], bins=[-1, 0, 10, 20, 30, 40, 51, 52], labels=[0, 1, 2, 3, 4, 5, 6])
    return train, test

def Working_Week_assign_cluster(label):
    if label < 1:
        return '0'
    elif label < 4:
        return '1_to_30'
    elif label < 6:
        return '31_to_51'
    else:
        return '52'
    
def apply_Working_Week(train, test):
    train, test = Working_Week_binning(train, test)
    train['Working_Week (Yearly)'] = train['Working_Week (Yearly)'].apply(Working_Week_assign_cluster)
    test['Working_Week (Yearly)'] = test['Working_Week (Yearly)'].apply(Working_Week_assign_cluster)
    return train, test

In [ ]:
# industry_status 제거 함수
def delete_industry(train, test):
    train = train.drop('Industry_Status', axis=1)
    test = test.drop('Industry_Status', axis=1)
    return train, test 

In [ ]:
# Occupation_Status 제거 함수
def delete_occupation(train, test):
    train = train.drop('Occupation_Status', axis=1)
    test = test.drop('Occupation_Status', axis=1)
    return train, test

In [ ]:
# race 변수 삭제 함수 
def delete_race(train, test):
    train = train.drop('Race', axis=1)
    test = test.drop('Race', axis=1)
    return train, test

In [ ]:
# Hispanic_Origin을 그룹화하는 함수 정의
def Hispanic_Origin_assign_cluster(status):
    if status == ' All other':
        return 'All other'
    else:
        return 'Not_All_other'


def apply_Hispanic_Origin(train, test):
    train['Hispanic_Origin'] = train['Hispanic_Origin'].apply(Hispanic_Origin_assign_cluster)
    test['Hispanic_Origin'] = test['Hispanic_Origin'].apply(Hispanic_Origin_assign_cluster)
    return train, test


In [ ]:
# Martial_Status를 그룹화하는 함수 정의
def Martial_Status_assign_cluster(status):
    # 최종 결정된 그룹들
    cluster_1 = ['Married', 'Divorced','Married (Armed Force Spouse)']
    cluster_2 = ['Separated', 'Widowed','Married (Spouse Absent)']
    if status in cluster_1:
        return 'cluster_1'
    elif status in cluster_2:
        return 'cluster_2'
    else:
        return 'Single'
    
def apply_Martial_Status(train, test):
    train['Martial_Status'] = train['Martial_Status'].apply(Martial_Status_assign_cluster)
    test['Martial_Status'] = test['Martial_Status'].apply(Martial_Status_assign_cluster)
    return train, test

In [ ]:
def Household_Summary_assign_cluster(status):
    # 최종 결정된 그룹들
    cluster_1 = ['Householder', 'Child under 18 ever married']
    cluster_2 = ['Other relative of householder','Nonrelative of householder', 'Group Quarters- Secondary individual']
    if status in cluster_1:
        return 'cluster_1'
    elif status in cluster_2:
        return 'cluster_2'
    else:
        return status

def apply_Household_Summary(train, test):
    train['Household_Summary'] = train['Household_Summary'].apply(Household_Summary_assign_cluster)
    test['Household_Summary'] = test['Household_Summary'].apply(Household_Summary_assign_cluster)
    return train, test

In [ ]:
# Citizenship을 그룹화하는 함수 정의
def Citizenship_assign_cluster(status):
    # 최종 결정된 그룹들
    cluster_1 = ['Native', 'Foreign-born (Naturalized US Citizen)']
    cluster_2 = ['Foreign-born (Non-US Citizen)', 'Native (Born Abroad)', 'Native (Born in Puerto Rico or US Outlying)']
    if status in cluster_1:
        return 'cluster_1'
    elif status in cluster_2:
        return 'cluster_2'
    else:
        return status
        
def apply_Citizenship(train, test):
    train['Citizenship'] = train['Citizenship'].apply(Citizenship_assign_cluster)
    test['Citizenship'] = test['Citizenship'].apply(Citizenship_assign_cluster)
    return train, test

In [ ]:
# Birth_Country의 범주를 그룹화하는 함수 정의
def Birth_Country_assign_cluster(status):
    if status == 'US':
        return 'US'
    else:
        return 'Not US'
    
def apply_Birth_Country(train, test):
    train['Birth_Country'] = train['Birth_Country'].apply(Birth_Country_assign_cluster)
    test['Birth_Country'] = test['Birth_Country'].apply(Birth_Country_assign_cluster)
    return train, test


In [ ]:
# Birth_Country (Father)의 범주를 그룹화하는 함수 정의
def Birth_Country_F_assign_cluster(status):
    if status == 'US':
        return 'US'
    else:
        return 'Not US'
    
def apply_Birth_F_Country(train, test):
    train['Birth_Country (Father)'] = train['Birth_Country (Father)'].apply(Birth_Country_F_assign_cluster)
    test['Birth_Country (Father)'] = test['Birth_Country (Father)'].apply(Birth_Country_F_assign_cluster)
    return train, test

In [ ]:
# Birth_Country (Mother)의 범주를 그룹화하는 함수 정의
def Birth_Country_M_assign_cluster(status):
    if status == 'US':
        return 'US'
    else:
        return 'Not US'
    
def apply_Birth_M_Country(train, test):
    train['Birth_Country (Mother)'] = train['Birth_Country (Mother)'].apply(Birth_Country_M_assign_cluster)
    test['Birth_Country (Mother)'] = test['Birth_Country (Mother)'].apply(Birth_Country_M_assign_cluster)
    return train, test

In [ ]:
# Birth_Country 관련 변수 삭제 함수
def delete_birth_country(train, test):
    train = train.drop(['Birth_Country', 'Birth_Country (Father)', 'Birth_Country (Mother)'], axis=1)
    test = test.drop(['Birth_Country', 'Birth_Country (Father)', 'Birth_Country (Mother)'], axis=1)
    return train, test

In [ ]:
# Tax_Status를 그룹화하는 함수 정의
def Tax_Status_assign_cluster(status):
    # 최종 결정된 그룹
    cluster_1 = ['Married Filling Jointly one over 65 & one under 65 (MFJ)','Married Filling Jointly both over 65 (MFJ)']
    if status in cluster_1:
        return 'cluster_1'
    else:
        return status
    
def apply_Tax_Status(train, test):
    train['Tax_Status'] = train['Tax_Status'].apply(Tax_Status_assign_cluster)
    test['Tax_Status'] = test['Tax_Status'].apply(Tax_Status_assign_cluster)
    return train, test

In [ ]:
# total 변수 그룹화
def total_assign_cluster(val):
    if val == 0:
        return 'zero'
    else:
        return 'non_zero'
    
def apply_total(train, test):
    train['total'] = train['total'].apply(total_assign_cluster)
    test['total'] = test['total'].apply(total_assign_cluster)
    return train, test

In [ ]:
# dividends 변수 그룹화
def dividends_assign_cluster(val):
    if val == 0:
        return 'zero'
    else:
        return 'non_zero'
    
def apply_dividends(train, test):
    train['Dividends'] = train['Dividends'].apply(dividends_assign_cluster)
    test['Dividends'] = test['Dividends'].apply(dividends_assign_cluster)
    return train, test

In [ ]:
# Income_Status 변수 삭제 함수
def delete_Income_Status(train, test):
    train = train.drop('Income_Status', axis=1)
    test = test.drop('Income_Status', axis=1)
    return train, test

## 모델 학습 관련 정의

In [ ]:
## 기본적인 모델들 돌려보기
from sklearn.linear_model import LinearRegression, ElasticNet, Lasso, Ridge
from sklearn.tree import DecisionTreeRegressor
from sklearn.ensemble import RandomForestRegressor
from sklearn.preprocessing import  LabelEncoder
import xgboost as xgb
import lightgbm as lgb
from sklearn.metrics import mean_squared_error
from catboost import CatBoostRegressor

# 모델 정의
reg = LinearRegression(n_jobs=-1)
ridge = Ridge(random_state = 42)
lasso = Lasso(random_state=42)
Enet = ElasticNet(random_state=42)
DTree = DecisionTreeRegressor(random_state=42)
rf = RandomForestRegressor(random_state=42, n_jobs=-1)
model_xgb = xgb.XGBRegressor(random_state=42, n_jobs=-1)
model_lgb = lgb.LGBMRegressor(random_state=42, n_jobs=-1)
model_cat = CatBoostRegressor(allow_writing_files=False, random_state=42, verbose=0)

In [ ]:
# 모델 학습을 위한 데이터 분리
train_X = train.drop(['ID', 'Income'], axis=1)
train_X.reset_index(drop=True, inplace=True)
train_y = train['Income']
train_y.reset_index(drop=True, inplace=True)

In [ ]:
def RMSE(y, y_pred):
    rmse = np.sqrt(mean_squared_error(y, y_pred))
    return rmse

# kfold 적용
from sklearn.model_selection import KFold

def rmse_cv(model):
    # cv별로 학습하는 함수
    kf = KFold(n_splits=5, shuffle=True, random_state=42)
    rmse_scores = []  
    model_name = model.__class__.__name__
    for train_index, test_index in kf.split(train_X):
      # train, test 데이터 분리
      X_train, X_test = train_X.iloc[train_index, :], train_X.iloc[test_index, :]
      y_train, y_test = train_y.iloc[train_index], train_y.iloc[test_index]

      # 모델 학습 및 예측
      reg = model.fit(X_train, y_train)
      pred = reg.predict(X_test)

    # 평가 지표 계산
      rmse = RMSE(y_test, pred)
      rmse_scores.append(rmse)

    return model_name, rmse_scores

def print_rmse_score(model):
    # cv별 프린팅, 평균 저장
    model_name, score = rmse_cv(model)
    # 출력 코드 제외 -> 필요시 주석 해제
    # for i, r in enumerate(score, start=1):
    #     print(f'{i} FOLDS: {model_name} RMSLE: {r:.4f}')
    # print(f'\n{model_name} mean RMSLE: {np.mean(score):.4f}')
    # print('='*40)
    return model_name, np.mean(score), np.std(score)

## 변수 제거 함수만 적용

- 걸과 : 변수 제거는 성능 향상에 큰 도움 되지 않았음

### 적용할 함수 경우의 수 생성

In [ ]:
delete_func = [delete_seek_employ, delete_industry, delete_occupation, delete_race, delete_birth_country, delete_Income_Status]

In [ ]:
# 해당 함수의 조합 생성
import itertools

# 가능한 모든 조합 구하기
all_combinations = list(itertools.chain.from_iterable(itertools.combinations(delete_func, r) for r in range(len(delete_func)+1)))

print(all_combinations)

In [ ]:
encode_list = []
comb_list = []
result_list = []
for i, comb in enumerate(all_combinations):
    print(f'{i}번째 조합')
    if len(comb) >= 2:
        break
    # 인코딩 함수 적용
    for encode in [apply_label_encoding, apply_one_hot_encoding, apply_target_encoding]:
        # 원본 데이터 불러오기
        train = pd.read_csv('../data/raw/train.csv')
        test = pd.read_csv('../data/raw/test.csv')
        # 전처리 함수 적용
        train, test = preprocess(train, test)

        # 각 조합에 맞는 함수 적용
        func_name_list = []
        for func in comb:
            func_name_list.append(func.__name__)
            train, test = func(train, test)
        print('적용중인 함수들:', func_name_list)

        print(encode.__name__, '진행중')
        comb_list.append(func_name_list)
        encode_list.append(encode.__name__)
        train, test = encode(train, test)
        
        # 모델 학습을 위한 데이터 분리
        train_X = train.drop(['ID', 'Income'], axis=1)
        train_X.reset_index(drop=True, inplace=True)
        train_y = train['Income']
        train_y.reset_index(drop=True, inplace=True)

        # 
        models = []
        scores = []
        std = []
        for model in [rf, model_xgb, model_lgb, model_cat]:
            model_name, mean_score, std_score = print_rmse_score(model)
            models.append(model_name)
            scores.append(mean_score)
            std.append(std_score)
        result_df = pd.DataFrame({'Model': models, 'Score': scores, 'Std' : std}).reset_index(drop=True)
        # 상위 3개 모델 print
        print(result_df.sort_values(by='Score').head(3))
        result_list.append(result_df)
final_result_df = pd.DataFrame({'Combination': comb_list, 'Encoding': encode_list, 'Result': result_list})

In [ ]:
final_result_df['Best_Score'] = final_result_df['Result'].apply(lambda x: x.sort_values(by='Score').head(1)['Score'].values[0])
final_result_df['Best_model'] = final_result_df['Result'].apply(lambda x: x.sort_values(by='Score').head(1)['Model'].values[0])
final_result_df.sort_values(by='Best_Score').head(5)

In [ ]:
final_result_df[final_result_df['Encoding'] == 'apply_target_encoding'].sort_values(by='Best_Score').head(5)

## 범주화 적용 성능 확인

In [ ]:
FE_functions = [apply_Education, apply_Employment, apply_Working_Week, apply_Hispanic_Origin, apply_Martial_Status,
                apply_Household_Summary, apply_Citizenship, apply_Tax_Status, apply_total, apply_dividends]

In [ ]:
encode_list = []
result_list = []
for encode in [apply_label_encoding, apply_one_hot_encoding, apply_target_encoding]:
    # 원본 데이터 불러오기
    train = pd.read_csv('../data/raw/train.csv')
    test = pd.read_csv('../data/raw/test.csv')
    # 전처리 함수 적용
    train, test = preprocess(train, test)

    for func in FE_functions:
        train, test = func(train, test)

    print(encode.__name__, '진행중')
    encode_list.append(encode.__name__)
    train, test = encode(train, test)
    
    # 모델 학습을 위한 데이터 분리
    train_X = train.drop(['ID', 'Income'], axis=1)
    train_X.reset_index(drop=True, inplace=True)
    train_y = train['Income']
    train_y.reset_index(drop=True, inplace=True)

    # 
    models = []
    scores = []
    std = []
    for model in [rf, model_xgb, model_lgb, model_cat]:
        model_name, mean_score, std_score = print_rmse_score(model)
        models.append(model_name)
        scores.append(mean_score)
        std.append(std_score)
    result_df = pd.DataFrame({'Model': models, 'Score': scores, 'Std' : std}).reset_index(drop=True)
    # 상위 3개 모델 print
    print(result_df.sort_values(by='Score').head(3))
    result_list.append(result_df)
cluster_final_result_df = pd.DataFrame({'Encoding': encode_list, 'Result': result_list})

In [ ]:
cluster_final_result_df['Best_Score'] = cluster_final_result_df['Result'].apply(lambda x: x.sort_values(by='Score').head(1)['Score'].values[0])
cluster_final_result_df['Best_model'] = cluster_final_result_df['Result'].apply(lambda x: x.sort_values(by='Score').head(1)['Model'].values[0])
cluster_final_result_df.sort_values(by='Best_Score')

### birth country 변수 범주화 추가

In [ ]:
# 적용할 함수 목록
FE_functions = [apply_Education, apply_Employment, apply_Working_Week, apply_Hispanic_Origin, apply_Martial_Status,
                apply_Household_Summary, apply_Citizenship, apply_Birth_Country, apply_Birth_F_Country, apply_Birth_M_Country,  
                apply_Tax_Status, apply_total, apply_dividends]

In [ ]:
encode_list = []
result_list = []
for encode in [apply_label_encoding, apply_one_hot_encoding, apply_target_encoding]:
    # 원본 데이터 불러오기
    train = pd.read_csv('../data/raw/train.csv')
    test = pd.read_csv('../data/raw/test.csv')
    # 전처리 함수 적용
    train, test = preprocess(train, test)

    for func in FE_functions:
        train, test = func(train, test)

    print(encode.__name__, '진행중')
    encode_list.append(encode.__name__)
    train, test = encode(train, test)
    
    # 모델 학습을 위한 데이터 분리
    train_X = train.drop(['ID', 'Income'], axis=1)
    train_X.reset_index(drop=True, inplace=True)
    train_y = train['Income']
    train_y.reset_index(drop=True, inplace=True)

    # 
    models = []
    scores = []
    std = []
    for model in [rf, model_xgb, model_lgb, model_cat]:
        model_name, mean_score, std_score = print_rmse_score(model)
        models.append(model_name)
        scores.append(mean_score)
        std.append(std_score)
    result_df = pd.DataFrame({'Model': models, 'Score': scores, 'Std' : std}).reset_index(drop=True)
    # 상위 3개 모델 print
    print(result_df.sort_values(by='Score').head(3))
    result_list.append(result_df)
add_cluster_final_result_df = pd.DataFrame({'Encoding': encode_list, 'Result': result_list})

In [ ]:
add_cluster_final_result_df['Best_Score'] = add_cluster_final_result_df['Result'].apply(lambda x: x.sort_values(by='Score').head(1)['Score'].values[0])
add_cluster_final_result_df['Best_model'] = add_cluster_final_result_df['Result'].apply(lambda x: x.sort_values(by='Score').head(1)['Model'].values[0])
add_cluster_final_result_df.sort_values(by='Best_Score')

# 제거, 범주 합치기

In [ ]:
# 해당 함수의 조합 생성
import itertools

# 가능한 모든 조합 구하기
all_combinations = list(itertools.chain.from_iterable(itertools.combinations([delete_Income_Status, delete_race] , r) for r in range(len([delete_Income_Status, delete_race, delete_birth_country] )+1)))

print(all_combinations)

In [ ]:
encode_list = []
comb_list = []
result_list = []
for i, comb in enumerate(all_combinations):
    print(f'{i}번째 조합')
    # 인코딩 함수 적용
    for encode in [apply_label_encoding, apply_one_hot_encoding, apply_target_encoding]:
        # 원본 데이터 불러오기
        train = pd.read_csv('../data/raw/train.csv')
        test = pd.read_csv('../data/raw/test.csv')
        # 전처리 함수 적용
        train, test = preprocess(train, test)

        # 각 조합에 맞는 함수 적용
        func_name_list = []
        for func in comb:
            func_name_list.append(func.__name__)
            train, test = func(train, test)
        print('적용중인 함수들:', func_name_list)
        
        for func in FE_functions:
            train, test = func(train, test)

        print(encode.__name__, '진행중')
        comb_list.append(func_name_list)
        encode_list.append(encode.__name__)
        train, test = encode(train, test)
        
        # 모델 학습을 위한 데이터 분리
        train_X = train.drop(['ID', 'Income'], axis=1)
        train_X.reset_index(drop=True, inplace=True)
        train_y = train['Income']
        train_y.reset_index(drop=True, inplace=True)

        # 
        models = []
        scores = []
        std = []
        for model in [rf, model_xgb, model_lgb, model_cat]:
            model_name, mean_score, std_score = print_rmse_score(model)
            models.append(model_name)
            scores.append(mean_score)
            std.append(std_score)
        result_df = pd.DataFrame({'Model': models, 'Score': scores, 'Std' : std}).reset_index(drop=True)
        # 상위 3개 모델 print
        print(result_df.sort_values(by='Score').head(3))
        result_list.append(result_df)
total_final_result_df = pd.DataFrame({'Combination': comb_list, 'Encoding': encode_list, 'Result': result_list})

In [ ]:
total_final_result_df['Best_Score'] = total_final_result_df['Result'].apply(lambda x: x.sort_values(by='Score').head(1)['Score'].values[0])
total_final_result_df['Best_model'] = total_final_result_df['Result'].apply(lambda x: x.sort_values(by='Score').head(1)['Model'].values[0])
total_final_result_df.sort_values(by='Best_Score').head(5)

In [ ]:
total_final_result_df.sort_values(by='Best_Score')

# 최종 모델 학습

## 최종 선택 전처리, feature engineering

In [ ]:
# 원본 데이터 불러오기
train = pd.read_csv('../data/raw/train.csv')
test = pd.read_csv('../data/raw/test.csv')
# 전처리 함수 적용
train, test = preprocess(train, test)

In [ ]:
# 적용할 feature engineering 함수 목록
FE_functions = [apply_Education, apply_Employment, apply_Working_Week, apply_Hispanic_Origin, apply_Martial_Status,
                apply_Household_Summary, apply_Citizenship,
                apply_Tax_Status, apply_total, apply_dividends]

for func in FE_functions:
    train, test = func(train, test)

In [ ]:
# 타겟 인코딩 적용
train, test = apply_target_encoding(train, test)

In [ ]:
# 모델 학습을 위한 데이터 분리
train_X = train.drop(['ID', 'Income'], axis=1)
train_X.reset_index(drop=True, inplace=True)
train_y = train['Income']
train_y.reset_index(drop=True, inplace=True)

In [ ]:
# 모델 구축
models = []
scores = []
std = []
for model in [rf, model_xgb, model_lgb, model_cat]:
    model_name, mean_score, std_score = print_rmse_score(model)
    models.append(model_name)
    scores.append(mean_score)
    std.append(std_score)
result_df = pd.DataFrame({'Model': models, 'Score': scores, 'Std' : std}).reset_index(drop=True)
result_df

In [ ]:
model_cat = CatBoostRegressor(allow_writing_files=False, random_state=42, verbose=0)
model_cat.fit(train_X, train_y)

In [ ]:
pred = model_cat.predict(test.drop('ID', axis=1))
pred
test['Income'] = pred
# submision과 test의 ID를 기준으로 merge
submission = pd.read_csv('../data/raw/sample_submission.csv')
submission = submission.drop('Income', axis=1)
submission = submission.merge(test[['ID', 'Income']], on='ID', how='left')
submission.fillna(0, inplace=True)

# 소득이 0보다 작은 값은 0으로 변환
submission.loc[submission['Income'] < 0, 'Income'] = 0
submission.to_csv('../outputs/final_cat_target.csv', index=False)


## 하이퍼 파라미터 튜닝

In [ ]:
# optuna를 이용한 하이퍼 파라미터 튜닝
import optuna
from optuna.samplers import TPESampler
from sklearn.metrics import log_loss
from sklearn.model_selection import train_test_split

X_train, X_val, y_train, y_val = train_test_split(train_X, train_y, test_size=0.2, random_state=42)

def objective(trial):
    params = {
        'iterations':trial.suggest_int("iterations", 4000, 25000),
        'od_wait':trial.suggest_int('od_wait', 500, 2300),
        'learning_rate' : trial.suggest_uniform('learning_rate',0.01, 1),
        'reg_lambda': trial.suggest_uniform('reg_lambda',1e-5,100),
        'subsample': trial.suggest_uniform('subsample',0,1),
        'random_strength': trial.suggest_uniform('random_strength',10,50),
        'depth': trial.suggest_int('depth',1, 15),
        'min_data_in_leaf': trial.suggest_int('min_data_in_leaf',1,30),
        'leaf_estimation_iterations': trial.suggest_int('leaf_estimation_iterations',1,15),
        'bagging_temperature' :trial.suggest_loguniform('bagging_temperature', 0.01, 100.00),
        'colsample_bylevel':trial.suggest_float('colsample_bylevel', 0.4, 1.0),
        }


    model = CatBoostRegressor(allow_writing_files=False, **params, random_state=42)

    model.fit(
        X_train,
        y_train,
        eval_set=[(X_val, y_val)],
        early_stopping_rounds=50,
        verbose=False)
        
    cat_pred = model.predict(X_val)
    rmse = RMSE(y_val, cat_pred)

    return rmse

sampler = TPESampler(seed=42)
study = optuna.create_study(
    study_name="cat_parameter_opt",
    direction="minimize",
    sampler=sampler)
study.optimize(objective, n_trials=50)
print("Best Score:", study.best_value)
print("Best trial:", study.best_trial.params)

In [ ]:
model_cat = CatBoostRegressor(allow_writing_files=False, **study.best_trial.params, random_state=42)
model_cat.fit(train_X, train_y)

In [ ]:
pred = model_cat.predict(test.drop('ID', axis=1))
pred
test['Income'] = pred
# submision과 test의 ID를 기준으로 merge
submission = pd.read_csv('../data/raw/sample_submission.csv')
submission = submission.drop('Income', axis=1)
submission = submission.merge(test[['ID', 'Income']], on='ID', how='left')
submission.fillna(0, inplace=True)

# 소득이 0보다 작은 값은 0으로 변환
submission.loc[submission['Income'] < 0, 'Income'] = 0
submission.to_csv('../outputs/final_cat_target_optuna.csv', index=False)

# SHAP 적용

- 하이퍼 파라미터 튜닝 전 모델에 적용(시간 문제)

## 타겟 인코딩

In [ ]:
# 원본 데이터 불러오기
train = pd.read_csv('../data/raw/train.csv')
test = pd.read_csv('../data/raw/test.csv')
# 전처리 함수 적용
train, test = preprocess(train, test)
# 적용할 feature engineering 함수 목록
FE_functions = [apply_Education, apply_Employment, apply_Working_Week, apply_Hispanic_Origin, apply_Martial_Status,
                apply_Household_Summary, apply_Citizenship,
                apply_Tax_Status, apply_total, apply_dividends]

for func in FE_functions:
    train, test = func(train, test)
# 타겟 인코딩 적용
train, test = apply_target_encoding(train, test)
# 모델 학습을 위한 데이터 분리
train_X = train.drop(['ID', 'Income'], axis=1)
train_X.reset_index(drop=True, inplace=True)
train_y = train['Income']
train_y.reset_index(drop=True, inplace=True)
model_cat = CatBoostRegressor(allow_writing_files=False, random_state=42, verbose=0)
model_cat.fit(train_X, train_y)

In [ ]:
import shap
ex = shap.TreeExplainer(model_cat)

shap_v = ex(train_X)

In [ ]:
shap.summary_plot(shap_v)

SHAP value가 낮을 수록 y값을 낮춘다 

-> 분류애서는 SHAP value가 낮으면 class 0에 가까워진다.

SHAP value가 음수인쪽에 빨간 점이 많다 -> 해당 컬럼의 값이 높을 수록 y값을 낮춘다.(class 0에 가까워진다.)

SHAP value가 음수인쪽에 파란 점이 많다 -> 해당 컬럼의 값이 낮을 수록 y값을 낮춘다.(class 0에 가까워진다.)

SHAP value가 0이 되는 지점을 기준으로 점의 색이 구분이 되면 그 변수 먼저 살펴볼 필요가 있다.

In [ ]:
shap.plots.bar(shap_v)

## 원핫 인코딩

In [ ]:
# 원본 데이터 불러오기
train = pd.read_csv('../data/raw/train.csv')
test = pd.read_csv('../data/raw/test.csv')
# 전처리 함수 적용
train, test = preprocess(train, test)
# 적용할 feature engineering 함수 목록
FE_functions = [apply_Education, apply_Employment, apply_Working_Week, apply_Hispanic_Origin, apply_Martial_Status,
                apply_Household_Summary, apply_Citizenship,
                apply_Tax_Status, apply_total, apply_dividends]

for func in FE_functions:
    train, test = func(train, test)
# 타겟 인코딩 적용
train, test = apply_one_hot_encoding(train, test)
# 모델 학습을 위한 데이터 분리
train_X = train.drop(['ID', 'Income'], axis=1)
train_X.reset_index(drop=True, inplace=True)
train_y = train['Income']
train_y.reset_index(drop=True, inplace=True)
model_cat = CatBoostRegressor(allow_writing_files=False, random_state=42, verbose=0)
model_cat.fit(train_X, train_y)

In [ ]:
import shap
ex_2 = shap.TreeExplainer(model_cat)

shap_v = ex_2(train_X)

In [ ]:
shap.summary_plot(shap_v)

In [ ]:
shap.plots.bar(shap_v)